In [ ]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

load_dotenv()  # 加载.env文件里的变量
# print(os.getenv("DEEPSEEK_API_KEY"))  # 现在可以正常读取了

llm = ChatOpenAI(
        model="deepseek-chat",  # 使用的模型名称，目前官方推荐用 'deepseek-chat'
        api_key=os.getenv("DEEPSEEK_API_KEY"),  # 你的 DeepSeek API Key
        base_url="https://api.deepseek.com/v1",  # DeepSeek API 地址
        temperature=0,
    )

In [ ]:
from typing import Optional, Union

from openai import BaseModel
from pydantic import Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage,AnyMessage,AIMessage

class WeatherInfo(BaseModel):
    location:str=Field(description="the location of weather info")

class UserInfo(BaseModel):
    """Extracted user information, such as name,age, phone,email"""
    name:str=Field(description="The name of user")
    age:Optional[int]=Field(description="The age of user")
    email:str=Field(description="The email of user")
    phone:Optional[str]=Field(description="The phone of user")

class RealTimeInfo(BaseModel):
    """fetch real time infor"""
    query:str=Field(description="the query to fetch real time info")

class ConversationalResponse(BaseModel):
    response:str=Field(description="chat response from LLM")    

class FinalResponse(BaseModel):
    final_output:Union[UserInfo,ConversationalResponse,WeatherInfo,RealTimeInfo]
    

parser=PydanticOutputParser(pydantic_object=FinalResponse)
prompt=ChatPromptTemplate.from_messages([
    ('system','解析用户输入并提取个人信息 {format_instructions}'),
    ('human','{query}')
])



In [ ]:
from langgraph.prebuilt import ToolNode


@tool(args_schema=RealTimeInfo)
def fetch_real_time_info(query):
    """fetch real time info based on query"""
    print('fetch_real_time_info',query)
    return {'messages':['小米汽车真是好，九八九八不得了']}

@tool(args_schema=WeatherInfo)
def get_weather_info(location):
    """get the weather info based on location."""
    print('get_weather_info',location)
    return {'messages':['天气晴']}

@tool(args_schema=UserInfo)
def insert_db(name, age, email, phone):
    """insert user info into db"""
    print('userInfo:', name, age, email, phone)
    return {'messages':['插入成功']}

tools=[fetch_real_time_info,get_weather_info,insert_db]
toolNode=ToolNode(tools)

In [ ]:
model_with_tools=llm.bind_tools(tools)


prompt=prompt.partial(format_instructions=parser.get_format_instructions())
structured_llm=prompt | llm | parser

In [ ]:
print(model_with_tools.kwargs)
model_with_tools.invoke('北京的天气').tool_calls


In [ ]:
from langgraph._internal._constants import CONF, CONFIG_KEY_RUNTIME
from langgraph.runtime import Runtime

def chat_with_model(state):
    messages=state['messages']
    response=structured_llm.invoke(messages)
    return {'messages':[AIMessage(content=str(response))],
            'structured_output':response.final_output}

def execute_tool(state):
    print('execute_tool',state)
    config = {CONF: {CONFIG_KEY_RUNTIME: Runtime()}}
    # result=model_with_tools.invoke(str(state['structured_output']))
    # print('execute_tool',model_with_tools.invoke(str(state['structured_output'])).tool_calls)
    # 非常重要，通过下面的代码，可以保证result中的tool_calls有参数，否则，有时候tool_calls中的args是空对象，导致报错
    result = model_with_tools.invoke(
        [HumanMessage(content=f"请调用对应工具，参数如下：{state['structured_output']}")]
    )
    response=toolNode.invoke({"messages":[result,state['messages']]},config)
    return {'messages':[response]}

def final_answer(state):
    messages=state['messages']
    response=llm.invoke(messages)
    print('final_answer',response)
    return {'messages':[response]}


In [ ]:
from typing import Any, List, TypedDict, Optional
from typing_extensions import Annotated
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage
import operator
from langgraph.graph import START, StateGraph, END



class AgentState(TypedDict):
    messages: Annotated[List[AnyMessage], operator.add]
    structured_output: Optional[Any]  # 存放 FinalResponse.final_output，和消息管道分离
    
def routing_function(state:AgentState):
    final_output=state['structured_output']
    if isinstance(final_output,ConversationalResponse):
        return 'final_answer'
    else:
        return 'execute_tool'

builder=StateGraph(AgentState)

builder.add_node('chat_with_model',chat_with_model)
builder.add_node('execute_tool',execute_tool)
builder.add_node('final_answer',final_answer)

builder.add_edge(START,'chat_with_model')
builder.add_conditional_edges('chat_with_model',routing_function,
                              {
                                  'final_answer':'final_answer',
                                  'execute_tool':'execute_tool'
                              })

graph=builder.compile()

In [ ]:
from IPython.display import display,Image

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
result=graph.invoke({'messages':[HumanMessage(content='小米汽车最新消息')]})
print(result)
# query="小米汽车最新消息"
# input_message={"query":[HumanMessage(content=query)]}

# result=graph.invoke(input_message)

In [ ]:
result=graph.invoke({'messages':[HumanMessage(content='大连的天气')]})
print(result)

In [ ]:
result=graph.invoke({'messages':[HumanMessage(content='我叫奥特曼，今年38岁，邮箱地址是aoteman@qq.com,电话是123123123')]})
print(result)

In [ ]:
result